# Week 6 - Rainbow DQN and BBF - Workbook

Nine graded exercises. We assemble the Rainbow recipe one component at a time, derive the C51 categorical projection, and end with a smoke ablation showing how each ingredient contributes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, torch, matplotlib.pyplot as plt
import _pkg_loader  # registers w06_project
from w06_project import agent as agent_mod, train as train_mod, solutions

## 1. Double DQN target (two stars)
Compute $y = r + (1-d)\gamma\; Q_{\theta^-}(s', \arg\max_{a'} Q_\theta(s', a'))$.

Online net picks the argmax (carries selection noise); target net evaluates the chosen action (independent noise) - Jensen's bias cancels.

In [ ]:
def double_dqn_target(reward, done, q_on_next, q_tg_next, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
r = torch.tensor([1.0]); d = torch.tensor([0.0])
q_on = torch.tensor([[5., 1.]]); q_tg = torch.tensor([[3., 9.]])
out = solutions.exercise_1_double_dqn_target(r, d, q_on, q_tg, 0.5)
assert torch.allclose(out, torch.tensor([1.0 + 0.5 * 3.0]))
print('exercise 1 OK')

## 2. Dueling head (two stars)
$Q(s,a) = V(s) + (A(s,a) - \bar A(s))$. The mean-subtraction is what makes this identifiable - without it $V$ and the advantage offset are degenerate.

In [ ]:
def dueling_head(v, a):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
v = torch.tensor([[1.0]]); a = torch.tensor([[2., 4., 6.]])  # mean = 4
out = solutions.exercise_2_dueling_head(v, a)
assert torch.allclose(out, torch.tensor([[-1., 1., 3.]]))
print('exercise 2 OK')

## 3. PER importance-sampling weight (two stars)
$w_i = (N \cdot P(i))^{-\beta}$, normalised by $\max_j w_j$ so $w_i \in [0, 1]$.

> **Hint**: $\beta \to 1$ removes the bias but raises variance; common schedule is $\beta: 0.4 \to 1$ linearly over training.

In [ ]:
def per_weight(prob, N, beta):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = np.array([0.5, 0.3, 0.2])
w = solutions.exercise_3_per_weight(p, N=10, beta=0.4)
assert w.max() == 1.0 and (w >= 0).all()
print('exercise 3 OK')

## 4. PER priority update (one star)
After a training step, the new priorities at the sampled indices are $|\delta_i| + \epsilon$.

In [ ]:
def priority_update(old_priorities, idx, td_errors, eps):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = np.ones(5)
out = solutions.exercise_4_priority_update(p, np.array([1, 3]), np.array([-2.0, 0.5]), eps=1e-6)
assert out[1] > 1.999
print('exercise 4 OK')

## 5. NoisyNet $f$ function (one star)
$f(x) = \text{sign}(x)\sqrt{|x|}$ - the factorised-Gaussian transform that lets $\epsilon^W_{ij} = f(\epsilon^{out}_i) f(\epsilon^{in}_j)$ behave like independent noise.

In [ ]:
def noisy_f(x):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
out = solutions.exercise_5_noisy_f(torch.tensor([-4., 0., 9.]))
assert torch.allclose(out, torch.tensor([-2., 0., 3.]))
print('exercise 5 OK')

## 6. C51 categorical projection (three stars)
Given a discrete return distribution `p` on a fixed support `support`, compute the projected target distribution $\Phi \mathcal{T} Z$ for one Bellman backup. This is the most subtle algorithm in this curriculum's first half - take your time.

> **Hint**: each transformed atom $\tilde z_j = \text{clip}(r + \gamma z_j, V_{min}, V_{max})$ scatters its mass to the two neighbouring grid atoms with linear-interpolation weights $u - b$ and $b - l$ where $b = (\tilde z - V_{min})/\Delta z$.

In [ ]:
def c51_project(p, support, reward, done, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = torch.tensor([[0.2, 0.3, 0.5]])
support = torch.tensor([-1., 0., 1.])
out = solutions.exercise_6_categorical_project(p, support, torch.tensor([0.1]), torch.tensor([0.0]), 0.9)
assert abs(out.sum().item() - 1.0) < 1e-5
print('exercise 6 OK')

## 7. n-step return (two stars)
$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + (1 - d_{t+n}) \gamma^n Q(s_{t+n}, a_{t+n})$.

In [ ]:
def nstep_return(rewards, q_bootstrap, gamma, n, done_last):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert solutions.exercise_7_nstep_return([1, 1, 1], 10.0, 1.0, 2, False) == 12.0
assert solutions.exercise_7_nstep_return([1, 1, 1], 10.0, 1.0, 2, True) == 2.0
print('exercise 7 OK')

## 8. BBF shrink-and-perturb (two stars)
$\theta \leftarrow \alpha \theta + (1 - \alpha)\theta_{init}$ with $\alpha \approx 0.5$. Reset Adam moments too in practice. This is the single new ingredient that lifts Rainbow to BBF.

In [ ]:
def shrink_perturb(theta, theta_init, alpha):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
th = torch.ones(3); th0 = torch.zeros(3)
assert torch.allclose(solutions.exercise_8_shrink_perturb(th, th0, 1.0), th)
assert torch.allclose(solutions.exercise_8_shrink_perturb(th, th0, 0.0), th0)
print('exercise 8 OK')

## 9. Smoke ablation (three stars, meaningful objective)
**Matching angle.** Run Rainbow-lite vs. no-PER vs. no-Noisy on LunarLander. The gap between configurations is exactly the kind of *uplift* you would measure when ablating a recommender system - the IS-weight derivation here is the same OPE math we will use in Week 11 to estimate the gap from logged data.

In [ ]:
res = solutions.exercise_9_smoke_ablate(n_steps=1500)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for (name, log), color in zip(res.items(), [RL_PALETTE['blue'], RL_PALETTE['orange'], RL_PALETTE['teal']]):
    rs = np.asarray(log.episode_returns)
    ax.plot(np.arange(len(rs)), rs, label=name, color=color, alpha=0.8)
ax.set_xlabel('episode'); ax.set_ylabel('return'); ax.legend(); ax.set_title('Smoke ablation')
fig.tight_layout(); plt.show()
print({k: v.meta['mean_last_10'] for k, v in res.items()})

## Reflection
1. Which of the Rainbow components addresses *exploration*? Which addresses *value-estimate bias*? Which addresses *sample efficiency*?
2. Why does Dueling's mean-subtraction outperform max-subtraction even though both are identifiability fixes?
3. In what scenario could PER *hurt* rather than help? (Hint: think about high-noise rewards.)
4. BBF resets are a heavy hammer - propose a milder intervention that still combats plasticity loss.